# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Ranking / scoring, built on top of binary classification.**

Lane 2's deliverable is a ranked review queue — 'which pages first' — which the framing skill's mapping table calls a ranking/scoring problem with Precision@K as the metric. The mechanism is a classifier producing `P(page is declining)`; the product is a sort by that score.

**Correction to my own first instinct:** I initially reasoned ranking 'only makes sense if decliners are rare.' The actual number below shows they're 54.2% of the dataset — a majority, not a minority. That changes *why* ranking is needed, not whether it is: with over half the pages technically 'declining' by this proxy label, an editor with limited hours can't review all of them anyway, and a plain count doesn't tell you which of the 16,000+ down-trending pages to look at *first*. So the case for ranking is capacity under abundance, not scarcity — a subtly different justification than the one I started with, and a good example of why I check the number before trusting the framing.

In [1]:
import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
down_share = (df['trend_direction'].str.lower()=='down').mean()*100
print(f"Share currently labeled declining: {down_share:.1f}%")
print("That's a majority, not a minority — so 'ranking is needed because decliners are rare' is wrong.")
print("The real justification: even within a majority-declining set, capacity is limited, so *ordering*")
print("by severity/confidence still matters more than a plain filter would.")

Share currently labeled declining: 54.2%
That's a majority, not a minority — so 'ranking is needed because decliners are rare' is wrong.
The real justification: even within a majority-declining set, capacity is limited, so *ordering*
by severity/confidence still matters more than a plain filter would.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target: `is_declining_label = (trend_direction == "down")`.**

This is a **proxy label, not an observed outcome** — and I want to be explicit about that rather than bury it. `trend_direction` is a bucket computed from the *current* window (it's derived from `trend_pct`, which is why `trend_pct` and `trend_direction` can never be features — they'd leak the answer into itself). The lane guide calls this exact label a 'beginner proxy label' on purpose: it shows the workflow end to end, but it isn't the ideal capstone target.

For this notebook I'm using it anyway, because the starter CSV is a single snapshot with no forward-looking window baked in. For the actual capstone, I plan to move to a genuinely observed future outcome from the warehouse release: features from a prior 90-day window predicting a decline measured in the *next* 30 days — an outcome that happens after the feature window closes, not a bucket computed from inside it.

In [2]:
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df["is_declining_label"].value_counts())
print(f"\nBase rate: {df['is_declining_label'].mean():.3f} — the number any dumb 'always guess majority' baseline gets for free.")

is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Base rate: 0.542 — the number any dumb 'always guess majority' baseline gets for free.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: Precision@50** (with Precision@20 reported alongside, since the two can disagree — I saw this happen in notebook 02, where the hand rule won at one K and the tree won at the other).

This is the metric that matches how the output actually gets used: an editor works down a queue with limited hours, not the full 30,000-row list. The lane guide's own baseline numbers give me something concrete to beat — the hand-written rule scores 0.240 at Precision@50 on this data, meaning about 12 of its top 50 picks are actually declining. 'Good' for my model means clearing that bar by a real margin, not by a rounding error, and I'll report the number of pages that represents (12 of 50, 30 of 50, whatever it turns out to be) rather than just the decimal — raw accuracy would hide this, since 54% of pages are already labeled 'down' and a model could look accurate while being useless at the top of the queue specifically.

In [3]:
baseline_precision_at_50 = 0.240  # from the lane guide's verified starter pipeline results
print(f"Baseline to beat: {baseline_precision_at_50:.3f} — about {round(baseline_precision_at_50*50)} of the top 50 picks correct.")
print("My target: a real, reportable improvement over this — not a rounding-error win.")

Baseline to beat: 0.240 — about 12 of the top 50 picks correct.
My target: a real, reportable improvement over this — not a rounding-error win.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one pseudonymized content page, summarized over its trailing 90 days**, for a client that has that page under active tracking. `content_id` and `client_id` are pseudonyms used only for grouping and holdout splits, never as features.

In [4]:
lane_cols = ["content_id", "client_id", "content_type", "main_intent",
             "impressions_90d", "sessions_90d", "avg_position", "ctr",
             "days_since_last_update", "content_age_days", "word_count",
             "trend_direction", "is_declining_label"]
lane_slice = df[lane_cols]
print(f"{len(lane_slice):,} rows, one per content page — unit of analysis for the ranking model.")
lane_slice.head(3)

30,000 rows, one per content page — unit of analysis for the ranking model.


,content_id,client_id,content_type,main_intent,impressions_90d,sessions_90d,avg_position,ctr,days_since_last_update,content_age_days,word_count,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,3803,17,10.6,0.76,20,187,3221.0,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,15320,9,20.3,0.05,25,445,2481.0,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,12581,11,36.5,0.09,20,141,3515.0,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Because no single signal, or simple AND of signals, separates decliners from the rest** — the individual correlations are all weak, but a model can combine several weak, tangled signals into something usable in a way a hand-written if/else can't. This is also exactly what I saw earlier: a strict AND-rule (stale ≥180 days AND impressions ≥500) matched only 17 out of 30,000 pages — useless as a queue, even though each condition alone is reasonable. That's the failure mode a fixed rule falls into: tighten it and you miss almost everyone; loosen it and it stops meaning anything.

In [5]:
feats = ["days_since_last_update", "impressions_90d", "avg_position",
         "ctr", "word_count", "content_age_days", "engagement_rate"]
print("Individual correlation with is_declining_label (each one, alone, barely moves the needle):\n")
for f in feats:
    corr = df[f].corr(df["is_declining_label"])
    print(f"  {f:25s} {corr:+.3f}")
print("\nNo single feature clears +/-0.20. The signal, if it's there at all, is spread thin across")
print("several weak, interacting features — which is the textbook case for a learned model over a rule.")

Individual correlation with is_declining_label (each one, alone, barely moves the needle):

  days_since_last_update    +0.081
  impressions_90d           -0.018
  avg_position              -0.029
  ctr                       -0.062
  word_count                +0.090
  content_age_days          -0.164
  engagement_rate           -0.013

No single feature clears +/-0.20. The signal, if it's there at all, is spread thin across
several weak, interacting features — which is the textbook case for a learned model over a rule.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.